# Bragg diffraction with the native Gaussian components

`BraggSample` expands a periodic thin specimen's complex transmission into a fixed
set of reciprocal orders. Each order becomes a tilted Gaussian, so one broad
incident Gaussian can form a diffraction pattern through the usual lens and
detector pipeline. No local Taylor approximation is used at this sample.

This example uses $T(x,y)=\exp[i\cos(2\pi x/d)]$. Its exact Fourier coefficients
are $c_h=i^h J_h(1)$ and its well-separated order intensities are $J_h(1)^2$.
The sample, lens and back focal plane are at $z=0,f,2f$, respectively; the spot
positions are $x_h=h\lambda f/d$.

The specimen is infinite and periodic; the incident Gaussian supplies finite
illumination. A projected crystal unit-cell transmission can replace this
analytic grating. This is a thin projected model, not dynamical diffraction.

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import jv

from temgym_core.components import BraggSample, Detector, Lens
from temgym_core.evaluate import evaluate_gaussians, gaussian_image
from temgym_core.gaussian import FreeSpacePropagator, make_gaussian
from temgym_core.run import run_to_end

## Define the unit-cell transmission

Samples are in `(y, x)` order, at `x=i*period_x/nx` and `y=j*period_y/ny`, without
repeating the endpoint. Supply the physical complex transmission `exp(1j*phase)`;
there is no legacy phase-to-path encoding here. The transmission is specified
at the chosen beam energy. Keep all lengths in the beam's coordinate unit.

Refine the unit-cell grid and the order cutoff independently. Omitted orders
are not renormalized. The grid must resolve the transmission, including any
harmonics introduced by exponentiating the phase.

In [ ]:
voltage = 200e3
period = 5e-9
focal_length = 2.3e-3
waist = 40e-9
max_order = 8
orders = np.arange(-max_order, max_order + 1)
orders_hk = np.column_stack((orders, np.zeros_like(orders)))

def unit_cell(samples):
    phase = np.cos(2 * np.pi * np.arange(samples) / samples)
    return np.broadcast_to(np.exp(1j * phase), (4, samples))

sample = BraggSample.from_unit_cell(
    unit_cell(128), periods_xy=(period, period), orders_hk=orders_hk, z=0.0,
)
refined = BraggSample.from_unit_cell(
    unit_cell(256), periods_xy=(period, period), orders_hk=orders_hk,
)
np.testing.assert_allclose(sample.coefficients, refined.coefficients, atol=1e-15)
np.testing.assert_allclose(sample.coefficients, (1j ** orders) * jv(orders, 1.0), atol=1e-15)

In [ ]:
# The displaced source also exercises the phase at each beam centre.
source = make_gaussian(
    x=0.37 * period, voltage=voltage, waist_x=waist, waist_y=waist,
)
lens = Lens(z=focal_length, focal_length=focal_length)
detector = Detector(
    z=2 * focal_length, pixel_size=(12e-9, 12e-9), shape=(64, 2048),
)
# Native batching supports the sample's changing beam count.
output = run_to_end(
    source, (sample, lens, detector),
    propagator=FreeSpacePropagator(include_axial_path=False),
)
image = np.asarray(gaussian_image(output, detector, samples_per_pixel=2))
spacing = float(source.wavelength) * focal_length / period
print(f"1 incident Gaussian -> {output.x.size} outgoing orders")
print(f"First-order spacing: {spacing * 1e6:.6f} micrometres")

## Independent field, flux and order-intensity checks

The specimen-plane comparison uses points away from the unit-cell sampling
knots. The detector checks use absolute integrated intensities, with no fitted
intensity scale. For this broad illumination the diffracted spots are separated.

In [ ]:
check_plane = Detector(z=0.0, pixel_size=(2e-9, 0.47e-9), shape=(32, 256))
incident = evaluate_gaussians(source, check_plane, method="scan")
actual = evaluate_gaussians(sample(source), check_plane, method="scan")
x = check_plane.coords[:, 0].reshape(check_plane.shape)
expected = incident * jnp.exp(1j * jnp.cos(2 * jnp.pi * x / period))
relative_field_error = float(jnp.linalg.norm(actual - expected) / jnp.linalg.norm(expected))
assert relative_field_error < 1e-8

incident_power = np.pi * waist**2 / 2
flux_ratio = image.sum() / incident_power
np.testing.assert_allclose(flux_ratio, 1.0, atol=1e-9, rtol=0)
coords_x = np.asarray(detector.coords).reshape(*detector.shape, 2)[..., 0]
print(f"Independent complex-field relative error: {relative_field_error:.3g}")
print(f"Transmitted/incident flux: {flux_ratio:.12f}")
print("Order   Measured fraction   Analytic fraction")
for h in range(-3, 4):
    mask = np.abs(coords_x - h * spacing) < spacing / 2
    measured = image[mask].sum() / incident_power
    analytic = jv(h, 1.0)**2
    np.testing.assert_allclose(measured, analytic, rtol=1e-8, atol=1e-12)
    print(f"{h:+3d}       {measured:.8f}          {analytic:.8f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
extent_um = np.asarray(detector.extent) * 1e6
axes[0].imshow(
    np.log10(image / image.max() + 1e-12), extent=extent_um,
    origin="lower", aspect="auto", cmap="inferno", vmin=-8, vmax=0,
)
axes[0].set(xlabel="x (micrometres)", ylabel="y (micrometres)",
            title="Diffraction intensity (log10, peak normalized)")
axes[1].semilogy(coords_x[0] * 1e6, image.sum(axis=0) / incident_power)
for h in range(-3, 4):
    axes[1].axvline(h * spacing * 1e6, color="0.6", linestyle="--", linewidth=0.7)
axes[1].set(xlabel="x (micrometres)", ylabel="Incident-power fraction per column",
            title="Integrated diffraction orders", xlim=(-4*spacing*1e6, 4*spacing*1e6),
            ylim=(1e-10, 1))
plt.show()